# 00 &middot; Start here

**Do this one first!** Two parts:

1. **Get on the leaderboard**. Build the simplest possible model and
   submit it!
2. **Look at the data**. What was measured, how it was transformed, which compounds came later, and how much they changed.


---
### Setup

In [ ]:
#@title installs and cloning git!
# Run me first.
%pip -q install rdkit pandas numpy scipy scikit-learn huggingface_hub fsspec
!git clone https://github.com/agura-alt/ai4chem_openadmet.git
%cd ai4chem_openadmet

In [ ]:
#@title imports!
import os, sys
SETUP_DIR = os.path.abspath("Setup")
os.path.isdir(SETUP_DIR) or sys.exit(f"No Setup dir at {SETUP_DIR}; cwd is {os.getcwd()}")

if SETUP_DIR not in sys.path:
    sys.path.insert(0, SETUP_DIR)

assert os.path.exists("Setup/common.py") and os.path.getsize("Setup/common.py") > 1000, (
    "common.py is missing or truncated. Upload it using the folder icon in the "
    "left sidebar, then re-run this cell.")

sys.modules.pop("common", None)            # force a fresh read
import common


### First: join the `OpenADMET_TeamFolders` shared drive

Everything you make today &mdash; your split, your predictions, your submission
files &mdash; lives in a folder named after your team inside the
**`OpenADMET_TeamFolders`** shared drive. The repo you just cloned is read-only
and vanishes when this runtime disconnects; that folder is what persists.

1. Go to [drive.google.com](https://drive.google.com) and click **Shared drives**
   in the left sidebar. You should see **`OpenADMET_TeamFolders`** listed. (If you
   were emailed an invitation, accept it first.)
2. If it is *not* listed, tell an instructor **which Google account you are using
   in Colab** and ask to be added as a **Contributor**. A "shared with me" link is
   not enough &mdash; you have to be a member of the shared drive for it to show up
   in Colab.
3. Run the next cell and click through the Drive mount prompt, signing in with
   **that same Google account**.

**Check the output of the next cell.** The folder it prints must start with

```
/content/drive/Shareddrives/OpenADMET_TeamFolders/
```

If it prints a path under `admet_hackathon/` instead, Drive is not connected and
**your work will be lost when the runtime disconnects** &mdash; fix the access
above and re-run before you continue.

In [ ]:
# Your workspace: your split, your predictions, your submissions all live here.
# Use the same name in every notebook today.

common.setup(pair="your-pair-name")

---
## 1. The data

Two files. `train` has molecules **and** measurements. `test` has molecules
only &mdash; the measurements are what you are predicting. We perform some log scaling when you call "load_train()" -- you can look at the raw training data if you like by setting `log_scale=False`!

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style("whitegrid"); sns.set_context("notebook")

train = common.load_train()          # log-scale columns, ready to model
test  = common.load_test()           # blinded: endpoint columns are empty
train_raw = common.load_train(log_scale=False)   # the assays in their own units
print(f"{len(train):,} train / {len(test):,} test")
train.head()

### 1.1 What the molecules look like

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

def show(df, n=6, title="", seed=0):
    sample = df.sample(n, random_state=seed)
    mols = [Chem.MolFromSmiles(s) for s in sample[common.SMILES_COL]]
    print(title)
    display(Draw.MolsToGridImage([mol for mol in mols if mol is not None],
                                 molsPerRow=3, subImgSize=(320, 280),
                                 legends=sample[common.ID_COL].tolist()))

show(train, 6, "TRAIN -- earlier compounds, with measurements") # you can change the number of molecules you want to visualize
show(test,  6, "TEST -- later compounds, measurements withheld")

### 1.2 The nine endpoints

The names of the endpoints are also stored in `common.ENDPOINTS`

| column | what it measures | |
|---|---|---|
| `LogD` | lipophilicity at pH 7.4 | partition between organic solvent and aqueous |
| `LogS` | kinetic solubility (KSOL) | solubility in aqueous buffer (affects dosage) |
| `Log_HLM_CLint` | human liver microsome clearance | how fast humans destroy it |
| `Log_MLM_CLint` | mouse liver microsome clearance | how fast the mouse model destroys it |
| `Log_Caco_Papp_AB` | passive permeability | can it cross the gut wall |
| `Log_Caco_ER` | efflux ratio | is it being actively pumped back out |
| `Log_Mouse_PPB` | plasma protein binding | how much drug binds plasma/is not free |
| `Log_Mouse_BPB` | brain protein binding | how much drug binds in brain/is not free |
| `Log_Mouse_MPB` | muscle protein binding | how much drug binds in muscle/is not free |


---
## 2. How you are scored

The leaderboard metric is **MA-RAE**: macro-averaged relative absolute error.

For one endpoint, RAE is your total absolute error divided by the total
absolute error of a model that always guesses the mean:

$$\mathrm{RAE} = \frac{\sum |y - \hat{y}|}{\sum |y - \bar{y}|}$$

which gives it a very convenient reading:

| RAE | meaning |
|---|---|
| 1.0 | as good as guessing the average **of the set you are scored on** |
| 0.5 | half the error of guessing (roughly the winning entry) |
| >1.0 | actively worse than guessing |

<br><br>

RAE is computed for each endpoint, and then MA-RAE is then the plain average of these values. **Why might we want to average the RAE of each endpoint as opposed to averaging the error across all endpoints together?**

### &#9654;&#65039; Predict first

**You are about to build a model that predicts the TRAINING mean for every molecule.**

**If it gets scored on the training data:** Will it score MA-RAE of exactly 1.0, a bit under, or a bit over?

**If it gets scored on unseen data:** Will it score MA-RAE of exactly 1.0, a bit under, or a bit over?

Write your answer here before running the next cell!

> `your prediction:`

---
## 3. The null model

The null model should predict the same number for every molecule: whatever that
endpoint averaged over the training set is.


In [ ]:
def get_mean(train_df):
    """Calculate each endpoint's mean value in train_df."""
    mean_vals = {}
    for endpoint in common.ENDPOINTS:
        ### TODO ###
        mean_vals[endpoint] = ...
        ### END TODO ###
    return mean_vals

def predict_mean(target_df, mean_vals):
    out = pd.DataFrame({"Molecule Name": target_df["Molecule Name"]})
    for endpoint in common.ENDPOINTS:
        out[endpoint] = mean_vals[endpoint]
    return out

train_means = get_mean(train)
test_preds = predict_mean(test, train_means)
test_preds.head()

In [ ]:
#@title Check your work { display-mode: "form" }
common.check_predictions(test_preds, against=test)      # right shape, no gaps

ok = True
for endpoint in common.ENDPOINTS:
    col, expected = test_preds[endpoint], float(train[endpoint].mean())
    if col.map(lambda values: values is Ellipsis).any():
        print(f"  {endpoint}: still has `...` in it"); ok = False
    elif col.isna().all():
        print(f"  {endpoint}: all NaN. The mean must come from the TRAINING set -- the "
              "test set has no measurements to average."); ok = False
    elif col.nunique(dropna=False) != 1:
        print(f"  {endpoint}: not constant ({col.min():.3f} to {col.max():.3f}). The null "
              "model gives every molecule the same number."); ok = False
    elif not np.isclose(float(col.iloc[0]), expected):
        print(f"  {endpoint}: you predicted {float(col.iloc[0]):.4f}, "
              f"the training mean is {expected:.4f}"); ok = False

print("All nine endpoints match the training mean." if ok else "Not yet -- see above.")

---
## 4. Submit
We've made our first model! Let's make a submission.

## A: The validation leaderboard!

You have 10 submissions to the actual test set -- but you'll probably want to evaluate more than 10 models. So, you will compare model performance on a *validation split* of your own design. That number is your *estimate* of your test set performance which you will submit alongside your predictions.

**Motivation:** being able to estimate your performance on unseen data is important -- it's what allows a model to be used to make key decisions prospectively.

The validation leaderboard ranks teams on

$$|\text{your validation estimate} - \text{your actual test score}|$$

That doesn't reward *being* accurate &mdash; it rewards *knowing how accurate you are*.

We give you a random split of 20% to start with, but if you want a better estimate for the rest of your modeling (...or for the leaderboard) explore `01_validation.ipynb`!

**The steps:**
1. Pick a validation split
2. Fit a model on the non-validation slice of the training data.
3. Score the model on the validation split

In [ ]:
# Step 1: Pick a validation split
SPLIT = "random"
fold, _ = common.load_split(train, name = SPLIT)
is_train, is_val = train[fold == "train"], train[fold == "val"]

# Step 2: Fit a model on the non validation slice of the training data
tr_mean = get_mean(is_train)

# Step 3: Score the model on the validation split
va_pred = predict_mean(is_val, tr_mean)
common.score(is_val, va_pred, "null-mean", SPLIT).round(3)

## NOTE: When you pass in the model name ("null-mean") and the split name to common.score,
##       common.score adds your score so you can look at it later!
## So, make sure your model name will let you identify your model later :)

In [ ]:
# Just for fun, let's look at our metrics when predicting on the train set
tr_pred = predict_mean(is_train, tr_mean)
common.score(is_train, tr_pred).round(3)

## NOTE: Since we didn't pass in the model name ("null-mean") and the split name to common.score,
##       common.score will not save this score for later!

### So it is 1.0 either way?

On the training data it is exactly 1.000 &mdash; it has to be, RAE is *defined*
as error relative to the mean-guesser. On your held-out compounds it is about 1.00 too.

**Is this going to be true for the actual test set too?** Let's find out!

## B: The accuracy leaderboard!

This is more straightforward -- we'll be seeing how the models perform measured by the MA-RAE across all endpoints and the MAE across each individual endpoint.

**The steps**
1. Refit the model on all the training data.
2. Predict on the test set
3. Prepare submission.
3. (Optional) Upload to "Submissions to Score" folder!

In [ ]:
# Step 1: Refit the model on all the training data.
mean_vals = get_mean(train) # *not* based on SPLIT!

# Step 2: Predict on the test set
test_preds = predict_mean(test, mean_vals)

# Step 3: Prepare submission
submission_path = common.prepare_submission(test_preds, "null-mean", "random")

## NOTE: make sure the model name and split are the same as when you called "common.score"!
## you can add an argument if you want to make a note like this:
## common.prepare_submission(test_preds, "null-mean", "random", why="because i said so")

### Now go drag it across

`prepare_submission` wrote one file -- predictions, with your metadata in the
`#` lines at the top -- into your pair's folder in the shared Drive. It printed
the full path above.

In Google Drive, open that folder and drag the file into the **Submissions To Score** folder. Only the submissions in these folders will be scored! The null model submission won't count to your 10, so give it a try to see if it works!


### Compare that to what you predicted

Did your holdout performance match the test set performance?

Nothing about your model changed between those two numbers &mdash; only which
molecules it was scored on. **Which of the two is the number you would report to a project team?** How can you get a better estimate before submitting to the leaderboard?


---
# Part 2 &middot; The data

## What was measured

In [ ]:
counts = train[common.ENDPOINTS].notna().sum().sort_values(ascending=False)
pct = (100 * counts / len(train)).round(1)
import pandas as pd
pd.DataFrame({"measured": counts, "% of molecules": pct})

In [ ]:
measured = train[common.ENDPOINTS].notna()
order = measured.sum(axis=1).sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(7, 8))
ax.imshow(measured.loc[order].to_numpy(), aspect="auto", interpolation="nearest",
          cmap="viridis")
ax.set_xticks(range(len(common.ENDPOINTS)))
ax.set_xticklabels(common.ENDPOINTS, rotation=90)
ax.set_ylabel("molecules (sorted by how much data they have)")
ax.set_title("What was actually measured")
plt.tight_layout(); plt.show()

print("assays measured per molecule:")
print(measured.sum(axis=1).value_counts().sort_index().to_string())

**Which compounds have the most data, and why might that be?**

What would go wrong if you trained only on the compounds that have all nine
measurements?

---
## The log transform

Below: the same data in the assay's own units, then log-transformed. Everything
except LogD gets the transform.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
raw_cols = [col for col in common.RAW_ENDPOINTS if col != "LogD"]
for ax, col in zip(axes.ravel(), raw_cols):
    values = pd.to_numeric(train_raw[col], errors="coerce").dropna()
    ax.hist(values, bins=50, color="#c44")
    ax.set_title(col, fontsize=9); ax.set_yticks([])
fig.suptitle("RAW units", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, col in zip(axes.ravel(), common.ENDPOINTS):
    values = train[col].dropna()
    ax.hist(values, bins=50, color="#468")
    ax.set_title(f"{col}  (n={len(values):,})", fontsize=9); ax.set_yticks([])
fig.suptitle("LOG scale", y=1.01)
plt.tight_layout(); plt.show()

**Which of these distributions is going to give a regression model
the most trouble? Why?**

---
## Which compounds came later?

The train/test split is **temporal**: you get early compounds, you predict late
ones. That is what a real project asks: "will next month's compounds behave as you say".

There is no date column. How can I tell which compounds got made earlier?

In [ ]:
print(train[common.ID_COL].head(3).to_list())
print(test[common.ID_COL].head(3).to_list())

# pull the number out of the ID
reg_train = train[common.ID_COL].str.extract(r"(\d+)", expand=False).astype(float)
reg_test  = test[common.ID_COL].str.extract(r"(\d+)", expand=False).astype(float)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(reg_train, bins=60, alpha=.65, label="train")
ax.hist(reg_test,  bins=60, alpha=.65, label="test")
ax.set_xlabel("number in the compound ID"); ax.set_ylabel("compounds")
ax.legend(); plt.tight_layout(); plt.show()

Compounds get an ID when they are **registered**, and registration
numbers only ever increase. So the number is a stand-in for synthesis date, and
that is the only time axis you have.

---
## Did the compounds change over the campaign?

A lead-optimisation program is people deliberately changing molecules. So the
late compounds may not be drawn from the same distribution as the early ones.

In [ ]:
early = train[reg_train <= reg_train.quantile(0.8)]
late  = train[reg_train >  reg_train.quantile(0.8)]

rows = []
for endpoint in common.ENDPOINTS:
    early_values, late_values = early[endpoint].dropna(), late[endpoint].dropna()
    if len(early_values) > 30 and len(late_values) > 30:
        rows.append({"endpoint": endpoint, "early mean": early_values.mean(), "late mean": late_values.mean(),
                     "shift (sd)": (late_values.mean() - early_values.mean()) / early_values.std(),
                     "n late": len(late_values)})
shift = pd.DataFrame(rows).set_index("endpoint").sort_values("shift (sd)")
shift.round(2)

**Which endpoints moved most, and which barely moved?**

Pick the largest one and argue for a chemical reason a medicinal chemistry team
would have pushed that property over the life of a program.

### Back to Part 1

Your validation set was drawn at **random** from the training compounds, so it
has the same distribution as what the model saw. The test set does not &mdash;
it is later in time.

That is the whole gap between your predicted score and your actual one.

**Which split would have predicted your leaderboard score better?**
`01_validation` lets you build it!